In [2]:
import json
import os

import numpy as np
import prettytable as pt
import pandas as pd
import pyarrow.parquet as pq
from scipy.optimize import curve_fit
from scipy.integrate import quad

import hist
# import matplotlib.pyplot as plt
# from sklearn.metrics import roc_curve, roc_auc_score
# from scipy.integrate import trapezoid

import eos_utils as eos

# Workspace packages
from HHtobbyy.event_discrimination.DFDataset import DFDataset
from HHtobbyy.workspace_utils import match_sample
from HHtobbyy.event_discrimination.models import map_model_to_Model


dfdataset_config = "root://cmseos.fnal.gov//store/group/lpcdihiggsboost/tsievert/HiggsDNA_parquet/DFDatasets/vManosv6/24_Manosv6_2026-06-17_13-17-58/dataset_config.json"
dfdataset = DFDataset(dfdataset_config)
# model, model_config = "MLP", {"output_dirpath": "/uscms/home/tsievert/nobackup/XHYbbgg/Model_Outputs/ManosMLPv6/2026-06-17_13-17-58", "activation_func": "ELU"}
# model = map_model_to_Model(model)(dfdataset, model_config)

/store/group/lpcdihiggsboost/tsievert/HiggsDNA_parquet/DFDatasets/vManosv6/24_Manosv6_2026-06-17_13-17-58/dataset_config.json


In [51]:
cats = {
    "vbfhh_cat1": {
        "AUX_DVBFHH": 0.774,  # >
    },
    "cat1": {
        "AUX_DggFHH": 0.99292177413867067,  # >
        "AUX_DnonRes": 0.42667485697897478,  # <
        "AUX_DRes": 0.90612735576941672,  # <
    },
    "cat2": {
        "AUX_DggFHH": 0.68115907008322718,  # >
        "AUX_DnonRes": 0.00064209200655934,  # <
        "AUX_DRes": 0.96341029269140221,  # <
    },
    "cat3" : {
        "AUX_DggFHH": 0.87036774207656309,  # >
        "AUX_DnonRes": 0.00221775294750575,  # <
        "AUX_DRes": 0.84544492824676509,  # <
    }
}
filters = [[(col, '>' if col.endswith('HH') else '<', cut) for col, cut in cat.items()] for name, cat in cats.items()]
print(filters)
print('-'*60)

manos_cats = {
    "vbfhh_cat1": {
        "is_VBFHH_sig_score": 0.774,  # >
    },
    "cat1": {
        "is_ggHH_sig_score": 0.99292177413867067,  # >
        "is_nonRes_bkg_score": 0.42667485697897478,  # <
        "is_Res_bkg_score": 0.90612735576941672,  # <
    },
    "cat2": {
        "is_ggHH_sig_score": 0.68115907008322718,  # >
        "is_nonRes_bkg_score": 0.00064209200655934,  # <
        "is_Res_bkg_score": 0.96341029269140221,  # <
    },
    "cat3" : {
        "is_ggHH_sig_score": 0.87036774207656309,  # >
        "is_nonRes_bkg_score": 0.00221775294750575,  # <
        "is_Res_bkg_score": 0.84544492824676509,  # <
    }
}
manos_filters = [[(col, '>' if 'HH' in col else '<', cut) for col, cut in cat.items()] for name, cat in manos_cats.items()]
print(manos_filters)
print('-'*60)

[[('AUX_DVBFHH', '>', 0.774)], [('AUX_DggFHH', '>', 0.9929217741386707), ('AUX_DnonRes', '<', 0.4266748569789748), ('AUX_DRes', '<', 0.9061273557694167)], [('AUX_DggFHH', '>', 0.6811590700832272), ('AUX_DnonRes', '<', 0.00064209200655934), ('AUX_DRes', '<', 0.9634102926914022)], [('AUX_DggFHH', '>', 0.8703677420765631), ('AUX_DnonRes', '<', 0.00221775294750575), ('AUX_DRes', '<', 0.8454449282467651)]]
------------------------------------------------------------
[[('is_VBFHH_sig_score', '>', 0.774)], [('is_ggHH_sig_score', '>', 0.9929217741386707), ('is_nonRes_bkg_score', '<', 0.4266748569789748), ('is_Res_bkg_score', '<', 0.9061273557694167)], [('is_ggHH_sig_score', '>', 0.6811590700832272), ('is_nonRes_bkg_score', '<', 0.00064209200655934), ('is_Res_bkg_score', '<', 0.9634102926914022)], [('is_ggHH_sig_score', '>', 0.8703677420765631), ('is_nonRes_bkg_score', '<', 0.00221775294750575), ('is_Res_bkg_score', '<', 0.8454449282467651)]]
----------------------------------------------------

In [23]:
filepaths = dfdataset.get_test_filepaths(0, regex='')
# filepaths = {'test': ['root://cmseos.fnal.gov//store/group/lpcdihiggsboost/tsievert/HiggsDNA_parquet/DFDatasets/vManosv6/24_Manosv6_2026-06-17_13-17-58/Run3_2024/sim/GluGlutoHH_kl-1p00_kt-1p00_c2-0p00/nominal/NOTAG_preprocessed_test0.parquet']}
test_pre = None
for filepath in filepaths['test']:
    try:
        eos_filepath = eos.load_file_eos(filepath)
        table = pq.read_table(eos_filepath, filters=filters)
        if test_pre is None: test_pre = table.to_pandas()
        else: test_pre = pd.concat([test_pre, table.to_pandas()])
        eos.delete_lockfile(eos_filepath)
    except:
        print('Failed loading: '+'/'.join(filepath.split('/')[-5:]))

dipho_mass_window = np.logical_and(test_pre['AUX_mass'].gt(100).to_numpy(), test_pre['AUX_mass'].lt(180).to_numpy())
pho_mva_cut = test_pre['AUX_pass_mva-0.7'].gt(0).to_numpy()
snt_cuts = np.logical_and(dipho_mass_window, pho_mva_cut)
test = test_pre.loc[snt_cuts]

Exception in thread Thread-317 (watch_tmp):
Traceback (most recent call last):
  File "/uscms/home/tsievert/nobackup/miniforge3/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/uscms/home/tsievert/nobackup/miniforge3/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/uscms_data/d3/tsievert/XHYbbgg/eos-utils/src/eos_utils/watch_tmp.py", line 25, in watch_tmp
    raise TimeoutError(f"ERROR: Tmp file watching for {tmp_filepath} timed out.")
TimeoutError: ERROR: Tmp file watching for .tmp_load--5817468176195017662.parquet timed out.


Failed loading: 24_Manosv6_2026-06-17_13-17-58/Run3_2025/sim/ttHtoGG/NOTAG_preprocessed_test0.parquet


In [24]:
expected_eras = {
    # sim
    'Run2_2016postVFP/sim', 'Run2_2016preVFP/sim', 'Run2_2017/sim', 'Run2_2018/sim',
    'Run3_2022/sim/postEE', 'Run3_2022/sim/preEE', 'Run3_2023/sim/postBPix', 'Run3_2023/sim/preBPix', 'Run3_2024/sim', 'Run3_2025/sim',
    # data
    # 'Run2_2016postVFP/data', 'Run2_2016preVFP/data', 'Run2_2017/data', 'Run2_2018/data',
    # 'Run3_2022/data', 'Run3_2023/data', 'Run3_2024/data', 'Run3_2025/data'
}
set_of_unique_sample_eras = set(pd.unique(test_pre['AUX_sample_era']).tolist())
assert expected_eras == set_of_unique_sample_eras, f"sample eras do not match expected (either missing eras or names changed): \n {expected_eras} \n vs. \n {set_of_unique_sample_eras}"
for sample_era in pd.unique(test_pre['AUX_sample_era']):
    set_of_unique_sample_names = set(pd.unique(test_pre.loc[test_pre['AUX_sample_era'].eq(sample_era), 'AUX_sample_name']).tolist())
    if 'data' in sample_era:
        expected_set = {'Data'}
    elif 'sim' in sample_era:
        expected_set = {
            'DDQCDGJets', 'GGJets', 'TTGG', 
            'GluGluHToGG', 'VBFHToGG', 'VHToGG', 'ttHToGG', 'bbHToGG',
            'GluGluToHH_kl1p00_kt1p00_c20p00', 'VBFToHH_cv1p00_c2v1p00_c31p00'
        }
        if '201' in sample_era: expected_set.remove('bbHToGG')
        elif '2022' in sample_era: expected_set.remove('VBFToHH_cv1p00_c2v1p00_c31p00')
        elif 'Run3_2023/sim/postBPix' in sample_era: expected_set.remove('DDQCDGJets')
        elif '2025' in sample_era: expected_set.remove('ttHToGG')
    else:
        raise Exception(f"Improper naming for era, expected to either contain \'sim\' or \'data\': {sample_era}")
    assert set_of_unique_sample_names == expected_set, f"{sample_era} - samples names do not match expected: \n {expected_set} \n vs. \n {set_of_unique_sample_names}"
    # for sample_name in pd.unique(test_pre.loc[test_pre['AUX_sample_era'].eq(sample_era), 'AUX_sample_name']):
    #     print('-'*60, '\n', sample_name)


vbf23_mask = np.logical_and(
    test['AUX_sample_era'].isin(['Run3_2023/sim/preBPix', 'Run3_2023/sim/postBPix']),
    test['AUX_sample_name'].eq('VBFHH')
)  # makeup for lack of 2022 VBFHH signal 
test.loc[vbf23_mask, 'AUX_eventWeight'] = 1.78 * test.loc[vbf23_mask, 'AUX_eventWeight']
ddqcd23pre_mask = np.logical_and(
    test['AUX_sample_era'].isin(['Run3_2023/sim/preBPix']),
    test['AUX_sample_name'].eq('DDQCDGJets')
)  # makeup for lack of 2023postBPix DDQCDGJets 
test.loc[vbf23_mask, 'AUX_eventWeight'] = 1.53 * test.loc[ddqcd23pre_mask, 'AUX_eventWeight']
tth25_mask = np.logical_and(
    test['AUX_sample_era'].isin(['Run3_2024/sim']),
    test['AUX_sample_name'].eq('ttHToGG')
)  # makeup for lack of 2025 ttH 
test.loc[tth25_mask, 'AUX_eventWeight'] = 2 * test.loc[tth25_mask, 'AUX_eventWeight']

ValueError: Must have equal len keys and value when setting with an iterable

In [52]:
manos_sample = pd.read_parquet('/uscms/home/tsievert/nobackup/XHYbbgg/v6_merged_scored_events.parquet', filters=manos_filters)
manos_sample

,sample,year,lumi,event,run,mass,nonResReg_vbfpair_dijet_mass_DNNreg,is_boosted,y_proba,weight,weight_central,weight_interference,weight_nominal,rel_xsec_weight,weight_tot,score,is_nonRes_bkg_score,is_Res_bkg_score,is_ggHH_sig_score,is_VBFHH_sig_score
0,GGJets,2016,154,153393,1,139.009520,116.035283,False,-99.0,1.885662e-08,0.937809,0.0,0.093781,0.031976,0.031976,"[0.00025212744, 0.0023625302, 0.995826, 0.0015...",0.000252,0.002363,0.995826,0.001559
1,GGJets,2016,378,377019,1,116.795369,126.166095,False,-99.0,1.323452e-08,0.658201,0.0,0.065820,0.022442,0.022442,"[0.00033492837, 0.005860736, 0.9607204, 0.0330...",0.000335,0.005861,0.960720,0.033084
2,GGJets,2016,2996,2995889,1,171.404487,123.668745,False,-99.0,4.207334e-08,1.046231,0.0,0.209246,0.071345,0.071345,"[0.0013932141, 0.010970196, 0.98656505, 0.0010...",0.001393,0.010970,0.986565,0.001072
3,GGJets,2016,3638,3637360,1,117.708165,121.701889,False,-99.0,2.201736e-08,1.095004,0.0,0.109500,0.037335,0.037335,"[0.0012790998, 0.008000273, 0.9754558, 0.01526...",0.001279,0.008000,0.975456,0.015265
4,GGJets,2016,14186,14185188,1,157.113169,81.037684,False,-99.0,2.071010e-08,1.029989,0.0,0.102999,0.035119,0.035119,"[0.0021840138, 0.03671899, 0.95624703, 0.00484...",0.002184,0.036719,0.956247,0.004850
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
903923,Data,2025,1615,3460835794,398027,160.663539,103.918762,False,-99.0,1.000000e+00,1.000000,NaN,NaN,NaN,1.000000,"[0.0017438009, 0.017420232, 0.9764721, 0.00436...",0.001744,0.017420,0.976472,0.004364
903924,Data,2025,176,422329771,398391,122.647004,107.717850,False,-99.0,1.000000e+00,1.000000,NaN,NaN,NaN,1.000000,"[0.0021457057, 0.015055073, 0.97905135, 0.0037...",0.002146,0.015055,0.979051,0.003748
903925,Data,2025,214,481050706,398802,154.170830,87.753624,False,-99.0,1.000000e+00,1.000000,NaN,NaN,NaN,1.000000,"[0.0018607636, 0.020208234, 0.9748511, 0.00307...",0.001861,0.020208,0.974851,0.003080
903926,Data,2025,704,1746269700,398680,117.180525,81.437622,False,-99.0,1.000000e+00,1.000000,NaN,NaN,NaN,1.000000,"[0.0018443306, 0.06794319, 0.92502016, 0.00519...",0.001844,0.067943,0.925020,0.005192


In [33]:
test_sig = test.loc[test['AUX_sample_name'].eq('GluGluToHH_kl1p00_kt1p00_c20p00') & test['AUX_sample_era'].eq('Run3_2024/sim')]

In [38]:
def apply_cuts(df: pd.DataFrame, cuts: dict, init_mask: None|np.ndarray=None):
    if init_mask is None: init_mask = np.ones(len(df), dtype=bool)
    for cut_name, cut in cuts.items():
        init_mask = np.logical_and(
            init_mask, df.loc[:, cut_name].gt(cut).to_numpy() 
            if 'HH' in cut_name else (
                df.loc[:, cut_name].lt(cut).to_numpy() 
                if 'D' in cut_name or 'score' in cut_name else 
                df.loc[:, cut_name].eq(cut).to_numpy()
            )
        )
    return init_mask
cat1_test_sig = test_sig.loc[apply_cuts(test_sig, cats['cat1'])]
cat1_test_sig

,eta,lead_eta,lead_phi,lead_mvaID,sublead_eta,sublead_phi,sublead_mvaID,nonResReg_vbfpair_lead_bjet_eta,nonResReg_vbfpair_lead_bjet_phi,nonResReg_vbfpair_lead_bjet_bTagWPL,...,AUX_nonResReg_vbfpair_HHbbggCandidate_mass,AUX_nonResReg_vbfpair_max_nonbjet_btag,AUX_pass_mva-0.7,AUX_nonResReg_vbfpair_resolved_BDT_mask,AUX_label1D,AUX_eventWeightTrain,AUX_DnonRes,AUX_DRes,AUX_DggFHH,AUX_DVBFHH
0,-1.298404,-1.967691,0.824803,0.089475,-1.234737,1.543527,0.381872,-0.431248,-0.769119,1.258086,...,496.108216,0.989624,1,False,2,0.000022,0.000278,0.004629,0.993048,0.002046
11,0.205454,0.423937,1.370450,-0.212640,-0.359401,-1.391589,0.551762,0.643332,-0.082838,1.258086,...,396.585798,0.519043,1,True,2,0.000052,0.000316,0.002484,0.994894,0.002307
12,-0.184096,-0.624009,-1.274383,0.522149,0.140061,-1.058245,0.509540,-0.240209,0.685368,1.258086,...,503.844023,0.996460,1,True,2,0.000028,0.000085,0.003709,0.994748,0.001458
17,-0.383465,-0.149539,1.520080,0.364584,-1.064645,1.580036,0.029058,-0.484075,0.032005,1.258086,...,517.242590,0.953125,1,True,2,0.000056,0.000173,0.001935,0.996052,0.001840
19,-0.785713,-0.951999,-1.582093,0.477888,-1.232642,-0.614831,0.208640,-1.434468,0.384296,1.258086,...,393.455958,0.998657,1,True,2,-0.000016,0.000503,0.003300,0.994184,0.002013
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29506,0.584178,0.853710,-0.513285,0.497219,0.655863,-1.172811,0.546172,-0.150883,1.030228,1.258086,...,536.121252,0.998413,1,True,2,0.000037,0.000122,0.002374,0.995050,0.002453
29511,-0.641196,-1.249338,0.667814,0.480889,-0.480843,0.352371,-0.893389,-0.257454,-1.100086,1.258086,...,541.529007,0.983643,1,False,2,0.000034,0.000186,0.003658,0.993899,0.002258
29520,0.770802,1.501468,-0.955498,-1.003196,0.556782,-1.199784,0.373561,1.235630,0.655131,1.258086,...,537.863736,0.995483,1,True,2,0.000055,0.000111,0.001500,0.995317,0.003071
29521,0.068526,-0.335856,1.168217,0.489648,0.648217,1.497211,-1.430388,0.670886,-0.279682,1.258086,...,436.237742,0.998779,1,True,2,0.000043,0.000229,0.002556,0.995494,0.001721


In [39]:
manos_sig = manos_sample.loc[manos_sample['sample'].eq('GluGluToHH_kl-1p00_kt-1p00_c2-0p00') & manos_sample['year'].eq(2024)
manos_cat1_sig = manos_sig.loc[apply_cuts(manos_sig, manos_cats['cat1'])]
manos_cat1_sig

,sample,year,lumi,event,run,mass,nonResReg_vbfpair_dijet_mass_DNNreg,is_boosted,y_proba,weight,weight_central,weight_interference,weight_nominal,rel_xsec_weight,weight_tot,score,is_nonRes_bkg_score,is_Res_bkg_score,is_ggHH_sig_score,is_VBFHH_sig_score
1,GluGluToHH_kl-1p00_kt-1p00_c2-0p00,2024,153,7154,1,123.036786,111.180941,False,-99.0,0.000002,0.325072,0.0,0.010758,0.000022,0.000022,"[0.00027772892, 0.0046289414, 0.9930479, 0.002...",0.000278,0.004629,0.993048,0.002046
19,GluGluToHH_kl-1p00_kt-1p00_c2-0p00,2024,178,8324,1,121.812230,122.636068,False,-99.0,0.000005,0.775575,0.0,0.025666,0.000052,0.000052,"[0.00031565057, 0.0024838592, 0.9948939, 0.002...",0.000316,0.002484,0.994894,0.002307
20,GluGluToHH_kl-1p00_kt-1p00_c2-0p00,2024,178,8326,1,122.340466,116.118666,False,-99.0,0.000003,0.425223,0.0,0.014072,0.000028,0.000028,"[8.487522e-05, 0.0037092764, 0.9947478, 0.0014...",0.000085,0.003709,0.994748,0.001458
27,GluGluToHH_kl-1p00_kt-1p00_c2-0p00,2024,187,8748,1,127.468867,118.929858,False,-99.0,0.000006,0.845682,0.0,0.027987,0.000057,0.000057,"[0.00017260779, 0.0019353109, 0.9960522, 0.001...",0.000173,0.001935,0.996052,0.001840
34,GluGluToHH_kl-1p00_kt-1p00_c2-0p00,2024,194,9086,1,121.601944,114.028840,False,-99.0,-0.000002,0.238456,0.0,-0.007891,-0.000016,-0.000016,"[0.0005030726, 0.003299693, 0.9941843, 0.00201...",0.000503,0.003300,0.994184,0.002013
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65765,GluGluToHH_kl-1p00_kt-1p00_c2-0p00,2024,7234,339960,1,124.323642,139.791342,False,-99.0,0.000004,0.560419,0.0,0.018546,0.000038,0.000038,"[0.00012217755, 0.0023741205, 0.9950506, 0.002...",0.000122,0.002374,0.995051,0.002453
65778,GluGluToHH_kl-1p00_kt-1p00_c2-0p00,2024,7565,355548,1,124.986364,110.253622,False,-99.0,0.000003,0.506867,0.0,0.016774,0.000034,0.000034,"[0.00018606793, 0.0036576404, 0.9938985, 0.002...",0.000186,0.003658,0.993899,0.002258
65801,GluGluToHH_kl-1p00_kt-1p00_c2-0p00,2024,7757,364576,1,125.010301,126.040223,False,-99.0,0.000006,0.822891,0.0,0.027232,0.000055,0.000055,"[0.00011149996, 0.0015002702, 0.995317, 0.0030...",0.000111,0.001500,0.995317,0.003071
65804,GluGluToHH_kl-1p00_kt-1p00_c2-0p00,2024,7799,366514,1,125.548579,118.947060,False,-99.0,0.000004,0.650667,0.0,0.021533,0.000044,0.000044,"[0.0002288539, 0.002555732, 0.995494, 0.001721...",0.000229,0.002556,0.995494,0.001721


In [43]:

print('All event #s the same: ',
    np.all(
        manos_cat1_sig['event'].to_numpy().astype(np.int_) 
        == cat1_test_sig['AUX_event'].to_numpy().astype(np.int_)
    )
)

print('All output scores the same (to float32 precision): ',
    np.all(
        np.isclose(
            manos_cat1_sig.loc[:, 
                ['is_nonRes_bkg_score', 'is_Res_bkg_score', 'is_ggHH_sig_score', 'is_VBFHH_sig_score']
            ].to_numpy().astype(np.float32),
            cat1_test_sig.loc[:, 
                ['AUX_DnonRes', 'AUX_DRes', 'AUX_DggFHH', 'AUX_DVBFHH']
            ].to_numpy().astype(np.float32)
        )
    )
)

print('Yield of SnT 24 signal: ', manos_cat1_sig['weight_tot'].sum())
print('Yield of FCP 24 signal: ', cat1_test_sig['AUX_eventWeight'].sum())

All event #s the same:  True
All output scores the same (to float32 precision):  True
Yield of SnT 24 signal:  0.4190844020082845
Yield of FCP 24 signal:  0.4165224825536524


In [26]:
#############################################################
# ASCii histogram for rapid plotting
def ascii_hist(x, bins=10, weights=None):
    N,X = np.histogram(x, bins=bins, weights=weights)
    width = 50
    nmax = np.max(N.max())
    for (xi, n) in zip(X,N):
        bar = '#'*int(n*width/nmax)
        xi = '{0: <8.4g}'.format(xi).ljust(10)
        print('{0}| {1}'.format(xi,bar))
def ascii_hist(x, bins=10, weights=None, fit=None):
    N,X = np.histogram(x, bins=bins, weights=weights)
    width = 50
    nmax = np.max([N.max(), fit.max()])
    if fit is None:
        for (xi, n) in zip(X,N):
            bar = '#'*int(n*width/nmax)
            xi = '{0: <8.4g}'.format(xi).ljust(10)
            print('{0}| {1}'.format(xi,bar))
    else:
        for (xi, n, fiti) in zip(X,N,fit):
            bar = '#'*int(n*width/nmax)
            if fiti > n: bar = bar + ' '*(int(fiti*width/nmax) - int(n*width/nmax)) + '+'
            else: bar = ''.join([bar[j] if j != int(fiti*width/nmax) else '+' for j in range(len(bar))])
            xi = '{0: <8.4g}'.format(xi).ljust(10)
            print('{0}| {1}'.format(xi,bar))

#############################################################
# Sideband fit functions for nonRes bkg estimaton
def exp_func(x, a, b):
    return a * np.exp(b * x)
def sd_hist(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float]):
    return hist.Hist(
        hist.axis.Regular(int((fit_bins[1]-fit_bins[0])//fit_bins[2]), fit_bins[0], fit_bins[1], name="var", growth=False, underflow=False, overflow=False), 
        storage='weight'
    ).fill(var=mass, weight=weight)
def exp_mass_fit(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float], sigma: bool=False):
    _hist_ = sd_hist(mass, weight, fit_bins)
    params, _ = curve_fit(
        exp_func, _hist_.axes.centers[0]-_hist_.axes.centers[0][0], _hist_.values(), p0=(_hist_.values()[0], -0.1), 
        sigma=np.where(_hist_.values() != 0, np.sqrt(_hist_.variances()), 0.76) if sigma else None
    )
    print(_hist_)
    ascii_hist(mass, bins=np.arange(fit_bins[0], fit_bins[1], fit_bins[2]), weights=weight, fit=exp_func(_hist_.axes.centers[0]-_hist_.axes.centers[0][0], a=params[0], b=params[1]))
    return _hist_, params
def est_yield(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float], sr_masscut: list[float], sigma: bool=False):
    _hist_, fit_params = exp_mass_fit(mass, weight, fit_bins, sigma=sigma)
    return quad(exp_func, sr_masscut[0]-_hist_.axes.centers[0][0], sr_masscut[1]-_hist_.axes.centers[0][0], args=tuple(fit_params))[0] / fit_bins[2]

In [46]:
data_samples = [sample_name for sample_name in sorted(pd.unique(test['AUX_sample_name']).tolist()) if match_sample(sample_name, ['Data']) is not None]
sideband_samples = [sample_name for sample_name in sorted(pd.unique(test['AUX_sample_name']).tolist()) if match_sample(sample_name, ['GJet', 'TTG']) is not None]
non_sideband_samples = [sample_name for sample_name in sorted(pd.unique(test['AUX_sample_name']).tolist()) if sample_name not in data_samples+sideband_samples]
print(f"All samples: {sorted(pd.unique(test['AUX_sample_name']).tolist())}")
print(f"Data samples: {data_samples}")
print(f"nonRes MC samples: {sideband_samples}")
print(f"Res/Signal MC samples: {non_sideband_samples}")


# field_names = ['Category'] + non_sideband_samples + ['nonRes SB fit', 'data SB fit', 's/b with nonRes fit', 's/b with data fit']
# field_names = ['Category'] + non_sideband_samples + ['nonRes SB fit', 'data SB fit']
# field_names = ['Category'] + non_sideband_samples + sideband_samples + sideband_samples + ['nonRes SB fit', 'N Data SB', 'data SB fit']

field_names = ['Category'] + non_sideband_samples + sideband_samples + ['single-H', 'non-Res', 'N Data SB']
table = pt.PrettyTable(field_names=field_names, float_format=".5")

not_prev_cut_mask = {}
for i, (name, cuts) in enumerate(cats.items()):
    # if 'VBF' in name: continue

    for eras in [['Run2_2016preVFP/sim', 'Run2_2016postVFP/sim', 'Run2_2016preVFP/data', 'Run2_2016postVFP/data'], ['Run2_2017/sim', 'Run2_2017/data'], ['Run2_2018/sim', 'Run2_2018/data'], ['Run3_2022/sim/preEE', 'Run3_2022/sim/postEE', 'Run3_2022/data'], ['Run3_2023/sim/preBPix', 'Run3_2023/sim/postBPix', 'Run3_2023/data'], ['Run3_2024/sim', 'Run3_2024/data'], ['Run3_2025/sim', 'Run3_2025/data'], ['tot']]:
        if 'tot' not in eras:
            era_mask = test['AUX_sample_era'].isin(eras); evtwt_factor = 1
            era_name = eras[-1].split('_')[-1].replace('/data', '').replace('postVFP', '')
        else:
            era_mask = np.ones(len(test), dtype=bool); evtwt_factor = 1
            era_name = 'tot'
        new_row = [name+' '+era_name]
        nonRes_sideband = pd.DataFrame({'mass': pd.Series(dtype='float'), 'weight_tot': pd.Series(dtype='float')})

        singleH_yield = 0.
        nonRes_yield = 0.

        for sample in non_sideband_samples+sideband_samples:

            sample_mask = np.logical_and(era_mask, test['AUX_sample_name'].eq(sample))
            if sample+era_name not in not_prev_cut_mask: not_prev_cut_mask[sample+era_name] = sample_mask

            pass_cut_mask = not_prev_cut_mask[sample+era_name]
            if 'boost' not in name:
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, np.logical_and(test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].gt(80), test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].lt(190))
                )
            
            pass_cut_mask = apply_cuts(test, cuts, init_mask=pass_cut_mask)
            pass_cut_sr_mask = np.logical_and(
                pass_cut_mask,
                np.logical_and(test.loc[:, 'AUX_mass'].gt(122.5).to_numpy(), test.loc[:, 'AUX_mass'].lt(127.).to_numpy())
            )
            new_row.append((evtwt_factor * test.loc[pass_cut_sr_mask, 'AUX_eventWeight']).sum())

            if 'H' in sample and 'HH' not in sample: singleH_yield += new_row[-1]
            elif 'H' not in sample: nonRes_yield += new_row[-1]

            not_prev_cut_mask[sample+era_name] = np.logical_and(not_prev_cut_mask[sample+era_name], ~pass_cut_mask)

        new_row.append(singleH_yield); new_row.append(nonRes_yield)

        # for sb_sample_name, sb_sample_arr in zip(['nonRes', 'data'], [sideband_samples, data_samples]):
        #     sb_mask = np.zeros(len(test), dtype=bool)
        #     for sample in sb_sample_arr: sb_mask = np.logical_or(sb_mask, test['AUX_sample_name'].eq(sample))
        #     if sb_sample_name not in not_prev_cut_mask: not_prev_cut_mask[sb_sample_name] = sb_mask

        #     pass_cut_mask = not_prev_cut_mask[sb_sample_name]
        #     if i != 0:
        #         pass_cut_mask = np.logical_and(
        #             pass_cut_mask, np.logical_and(test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].gt(80), test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].lt(190))
        #         )
        #     for cut_name, cut in cuts.items():
        #         pass_cut_mask = np.logical_and(
        #             pass_cut_mask, test.loc[:, cut_name].gt(cut).to_numpy() 
        #             if 'HH' in cut_name else (
        #                 test.loc[:, cut_name].lt(cut).to_numpy() 
        #                 if 'D' in cut_name else 
        #                 test.loc[:, cut_name].eq(cut).to_numpy()
        #             )
        #         )
        #     pass_cut_sb_mask = np.logical_and(
        #         pass_cut_mask,
        #         np.logical_or(test.loc[:, 'AUX_mass'].lt(115.).to_numpy(), test.loc[:, 'AUX_mass'].gt(135.).to_numpy())
        #     )
        #     if sb_sample_name == 'data':
        #         sb_window_mask = np.logical_and(test.loc[:, 'AUX_mass'].gt(100.).to_numpy(), test.loc[:, 'AUX_mass'].lt(180.).to_numpy())
        #         new_row.append(np.sum(np.logical_and(pass_cut_sb_mask, sb_window_mask)))
        #     if np.sum(pass_cut_sb_mask) != 0:
        #         sb_est_yield = est_yield(test.loc[pass_cut_sb_mask, 'AUX_mass'], test.loc[pass_cut_sb_mask, 'AUX_eventWeight'], [100., 180., 5.], [115., 135.])
        #     else:
        #         sb_est_yield = 0
        #     new_row.append(sb_est_yield)

        for sb_sample_name, sb_sample_arr in zip(['data'], [data_samples]):
            sb_mask = np.logical_and(era_mask, test['AUX_sample_name'].isin(sb_sample_arr))
            if sb_sample_name+era_name not in not_prev_cut_mask: not_prev_cut_mask[sb_sample_name+era_name] = sb_mask

            pass_cut_mask = not_prev_cut_mask[sb_sample_name+era_name]
            if 'boost' not in name:
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, np.logical_and(test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].gt(80), test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].lt(190))
                )
            pass_cut_mask = apply_cuts(test, cuts, init_mask=pass_cut_mask)
            pass_cut_sb_mask = np.logical_and(
                pass_cut_mask,
                np.logical_or(test.loc[:, 'AUX_mass'].lt(120.).to_numpy(), test.loc[:, 'AUX_mass'].gt(130.).to_numpy())
            )
            sb_window_mask = np.logical_and(test.loc[:, 'AUX_mass'].gt(100.).to_numpy(), test.loc[:, 'AUX_mass'].lt(180.).to_numpy())
            new_row.append(np.sum(np.logical_and(pass_cut_sb_mask, sb_window_mask)))

        # sum_singleH = new_row[1] + sum(new_row[4:11])
        # new_row.append(new_row[2] / (sum_singleH + new_row[11]))
        # new_row.append(new_row[2] / (sum_singleH + new_row[12]))

        table.add_row(new_row)

print(table)


All samples: ['DDQCDGJets', 'GGJets', 'GluGluHToGG', 'GluGluToHH_kl1p00_kt1p00_c20p00', 'TTGG', 'VBFHToGG', 'VBFToHH_cv1p00_c2v1p00_c31p00', 'VHToGG', 'bbHToGG', 'ttHToGG']
Data samples: []
nonRes MC samples: ['DDQCDGJets', 'GGJets', 'TTGG']
Res/Signal MC samples: ['GluGluHToGG', 'GluGluToHH_kl1p00_kt1p00_c20p00', 'VBFHToGG', 'VBFToHH_cv1p00_c2v1p00_c31p00', 'VHToGG', 'bbHToGG', 'ttHToGG']
+-----------------+-------------+---------------------------------+----------+-------------------------------+----------+---------+----------+------------+----------+---------+----------+----------+-----------+
|     Category    | GluGluHToGG | GluGluToHH_kl1p00_kt1p00_c20p00 | VBFHToGG | VBFToHH_cv1p00_c2v1p00_c31p00 |  VHToGG  | bbHToGG | ttHToGG  | DDQCDGJets |  GGJets  |   TTGG  | single-H | non-Res  | N Data SB |
+-----------------+-------------+---------------------------------+----------+-------------------------------+----------+---------+----------+------------+----------+---------+---------

In [53]:
data_samples = [sample_name for sample_name in sorted(pd.unique(manos_sample['sample']).tolist()) if match_sample(sample_name, ['Data']) is not None]
sideband_samples = [sample_name for sample_name in sorted(pd.unique(manos_sample['sample']).tolist()) if match_sample(sample_name, ['GJet', 'TTG']) is not None]
non_sideband_samples = [sample_name for sample_name in sorted(pd.unique(manos_sample['sample']).tolist()) if sample_name not in data_samples+sideband_samples]
print(f"All samples: {sorted(pd.unique(manos_sample['sample']).tolist())}")
print(f"Data samples: {data_samples}")
print(f"nonRes MC samples: {sideband_samples}")
print(f"Res/Signal MC samples: {non_sideband_samples}")


# field_names = ['Category'] + non_sideband_samples + ['nonRes SB fit', 'data SB fit', 's/b with nonRes fit', 's/b with data fit']
# field_names = ['Category'] + non_sideband_samples + ['nonRes SB fit', 'data SB fit']
# field_names = ['Category'] + non_sideband_samples + sideband_samples + sideband_samples + ['nonRes SB fit', 'N Data SB', 'data SB fit']

field_names = ['Category'] + non_sideband_samples + sideband_samples + ['single-H', 'non-Res', 'N Data SB']
table = pt.PrettyTable(field_names=field_names, float_format=".5")

not_prev_cut_mask = {}
for i, (name, cuts) in enumerate(manos_cats.items()):
    # if 'VBF' in name: continue

    for eras in [[2016], [2017], [2018], [2022], [2023], [2024], [2025], ['tot']]:
        if 'tot' not in eras:
            era_mask = manos_sample['year'].isin(eras); evtwt_factor = 1
            era_name = str(eras[0])
        else:
            era_mask = np.ones(len(manos_sample), dtype=bool); evtwt_factor = 1
            era_name = 'tot'
        new_row = [name+' '+era_name]
        nonRes_sideband = pd.DataFrame({'mass': pd.Series(dtype='float'), 'weight_tot': pd.Series(dtype='float')})

        singleH_yield = 0.
        nonRes_yield = 0.

        for sample in non_sideband_samples+sideband_samples:

            sample_mask = np.logical_and(era_mask, manos_sample['sample'].eq(sample))
            if sample+era_name not in not_prev_cut_mask: not_prev_cut_mask[sample+era_name] = sample_mask

            pass_cut_mask = not_prev_cut_mask[sample+era_name]
            if 'boost' not in name:
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, np.logical_and(manos_sample.loc[:, 'nonResReg_vbfpair_dijet_mass_DNNreg'].gt(80), manos_sample.loc[:, 'nonResReg_vbfpair_dijet_mass_DNNreg'].lt(190))
                )
            pass_cut_mask = apply_cuts(manos_sample, cuts, init_mask=pass_cut_mask)
            pass_cut_sr_mask = np.logical_and(
                pass_cut_mask,
                np.logical_and(manos_sample.loc[:, 'mass'].gt(122.5).to_numpy(), manos_sample.loc[:, 'mass'].lt(127.).to_numpy())
            )
            new_row.append((evtwt_factor * manos_sample.loc[pass_cut_sr_mask, 'weight_tot']).sum())

            if 'H' in sample and 'HH' not in sample: singleH_yield += new_row[-1]
            elif 'H' not in sample: nonRes_yield += new_row[-1]

            not_prev_cut_mask[sample+era_name] = np.logical_and(not_prev_cut_mask[sample+era_name], ~pass_cut_mask)

        new_row.append(singleH_yield); new_row.append(nonRes_yield)

        # for sb_sample_name, sb_sample_arr in zip(['nonRes', 'data'], [sideband_samples, data_samples]):
        #     sb_mask = np.zeros(len(test), dtype=bool)
        #     for sample in sb_sample_arr: sb_mask = np.logical_or(sb_mask, test['AUX_sample_name'].eq(sample))
        #     if sb_sample_name not in not_prev_cut_mask: not_prev_cut_mask[sb_sample_name] = sb_mask

        #     pass_cut_mask = not_prev_cut_mask[sb_sample_name]
        #     if i != 0:
        #         pass_cut_mask = np.logical_and(
        #             pass_cut_mask, np.logical_and(test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].gt(80), test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].lt(190))
        #         )
        #     for cut_name, cut in cuts.items():
        #         pass_cut_mask = np.logical_and(
        #             pass_cut_mask, test.loc[:, cut_name].gt(cut).to_numpy() 
        #             if 'HH' in cut_name else (
        #                 test.loc[:, cut_name].lt(cut).to_numpy() 
        #                 if 'D' in cut_name else 
        #                 test.loc[:, cut_name].eq(cut).to_numpy()
        #             )
        #         )
        #     pass_cut_sb_mask = np.logical_and(
        #         pass_cut_mask,
        #         np.logical_or(test.loc[:, 'AUX_mass'].lt(115.).to_numpy(), test.loc[:, 'AUX_mass'].gt(135.).to_numpy())
        #     )
        #     if sb_sample_name == 'data':
        #         sb_window_mask = np.logical_and(test.loc[:, 'AUX_mass'].gt(100.).to_numpy(), test.loc[:, 'AUX_mass'].lt(180.).to_numpy())
        #         new_row.append(np.sum(np.logical_and(pass_cut_sb_mask, sb_window_mask)))
        #     if np.sum(pass_cut_sb_mask) != 0:
        #         sb_est_yield = est_yield(test.loc[pass_cut_sb_mask, 'AUX_mass'], test.loc[pass_cut_sb_mask, 'AUX_eventWeight'], [100., 180., 5.], [115., 135.])
        #     else:
        #         sb_est_yield = 0
        #     new_row.append(sb_est_yield)

        for sb_sample_name, sb_sample_arr in zip(['data'], [data_samples]):
            sb_mask = np.logical_and(era_mask, manos_sample['sample'].isin(sb_sample_arr))
            if sb_sample_name+era_name not in not_prev_cut_mask: not_prev_cut_mask[sb_sample_name+era_name] = sb_mask

            pass_cut_mask = not_prev_cut_mask[sb_sample_name+era_name]
            if 'boost' not in name:
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, np.logical_and(manos_sample.loc[:, 'nonResReg_vbfpair_dijet_mass_DNNreg'].gt(80), manos_sample.loc[:, 'nonResReg_vbfpair_dijet_mass_DNNreg'].lt(190))
                )
            pass_cut_mask = apply_cuts(manos_sample, cuts, init_mask=pass_cut_mask)
            pass_cut_sb_mask = np.logical_and(
                pass_cut_mask,
                np.logical_or(manos_sample.loc[:, 'mass'].lt(120.).to_numpy(), manos_sample.loc[:, 'mass'].gt(130.).to_numpy())
            )
            sb_window_mask = np.logical_and(manos_sample.loc[:, 'mass'].gt(100.).to_numpy(), manos_sample.loc[:, 'mass'].lt(180.).to_numpy())
            new_row.append(np.sum(np.logical_and(pass_cut_sb_mask, sb_window_mask)))

        # sum_singleH = new_row[1] + sum(new_row[4:11])
        # new_row.append(new_row[2] / (sum_singleH + new_row[11]))
        # new_row.append(new_row[2] / (sum_singleH + new_row[12]))

        table.add_row(new_row)

print(table)


All samples: ['DDQCDGJets', 'Data', 'GGJets', 'GluGluHToGG', 'GluGluToHH_kl-1p00_kt-1p00_c2-0p00', 'TTGG', 'VBFHH_CV-1p000_C2V-1p000_C3-1p000', 'VBFHToGG', 'VHToGG', 'ttHToGG']
Data samples: ['Data']
nonRes MC samples: ['DDQCDGJets', 'GGJets', 'TTGG']
Res/Signal MC samples: ['GluGluHToGG', 'GluGluToHH_kl-1p00_kt-1p00_c2-0p00', 'VBFHH_CV-1p000_C2V-1p000_C3-1p000', 'VBFHToGG', 'VHToGG', 'ttHToGG']
+-----------------+-------------+------------------------------------+-----------------------------------+----------+----------+----------+------------+---------+---------+----------+----------+-----------+
|     Category    | GluGluHToGG | GluGluToHH_kl-1p00_kt-1p00_c2-0p00 | VBFHH_CV-1p000_C2V-1p000_C3-1p000 | VBFHToGG |  VHToGG  | ttHToGG  | DDQCDGJets |  GGJets |   TTGG  | single-H | non-Res  | N Data SB |
+-----------------+-------------+------------------------------------+-----------------------------------+----------+----------+----------+------------+---------+---------+----------+----

In [ ]:
manos_sample = pd.read_parquet('/uscms/home/tsievert/nobackup/XHYbbgg/v6_merged_scored_events.parquet')
manos_sample['lumievent'] = (manos_sample['lumi'].astype(int).astype(str) + manos_sample['event'].astype(int).astype(str)).astype(int)
manos_sample.keys()

In [ ]:
for sample in pd.unique(manos_sample['sample']):
    sample_mask = manos_sample['sample'].eq(sample)
    # for year in pd.unique(manos_sample['year']):
    #     year_mask = manos_sample['year'].eq(year)
    #     print(sample, ' ', year, ' ', len(manos_sample.loc[np.logical_and(sample_mask, year_mask)]))
    print('num events of ', sample, ' ', 'all years', ' ', len(manos_sample.loc[sample_mask]))
    print('-'*60)

In [ ]:
for sample in pd.unique(test['AUX_sample_name']):
    sample_mask = test['AUX_sample_name'].eq(sample)
    # for year, year_tags in zip(['2016', '2017', '2018', '2022', '2023', '2024', '2025'], [['Run2_2016preVFP/sim', 'Run2_2016postVFP/sim', 'Run2_2016preVFP/data', 'Run2_2016postVFP/data'], ['Run2_2017/sim', 'Run2_2017/data'], ['Run2_2018/sim', 'Run2_2018/data'], ['Run3_2022/sim/preEE', 'Run3_2022/sim/postEE', 'Run3_2022/data'], ['Run3_2023/sim/preBPix', 'Run3_2023/sim/postBPix', 'Run3_2023/data'], ['Run3_2024/sim', 'Run3_2024/data'], ['Run3_2025/sim', 'Run3_2025/data']]):
    #     year_mask = test['AUX_sample_era'].isin(year_tags)
    #     print(sample, ' ', year, ' ', len(test.loc[np.logical_and(sample_mask, year_mask)]))
    print('num events of ', sample, ' ', 'all years', ' ', len(test.loc[sample_mask]))
    print('-'*60)

In [ ]:
import awkward as ak
import numba as nb

@nb.njit
def issubset(subset_candidate_arr, superset_candidate_arr, nonsubset_builder):
    nonsubset_builder.begin_list()
    for sub_elem in subset_candidate_arr:
        found = False
        for super_elem in superset_candidate_arr:
            found = found or (sub_elem == super_elem)
        nonsubset_builder.append(sub_elem * found)
    nonsubset_builder.end_list()
    return nonsubset_builder

In [ ]:
nonsubset_events = issubset(test['lumievent'].to_numpy(), manos_sample['lumievent'].to_numpy(), ak.ArrayBuilder()).snapshot()
# nonsubset_events